### AI Financial Analyst

#### Member 2

### Notebook 1: Knowledge Base Setup

---

#### Project Overview

The structured financial datasets have already been prepared by **Member 1**.
Before collecting SEC 10-K filings, we must verify that these datasets are
complete, consistent, and ready to be integrated into the Retrieval-Augmented
Generation (RAG) pipeline.

---

#### Tasks Performed

- Import required libraries
- Load structured datasets
- Standardize column names
- Explore dataset structure
- Validate dataset quality
- Check duplicates
- Check missing values
- Verify company-year coverage
- Generate project summary

---

#### CRISP-DM Phase

**Data Understanding**

This notebook corresponds to the Data Understanding phase of the
CRISP-DM methodology.

In [1]:
# Importing Required Libraries

import os
import pandas as pd
print("Libraries imported successfully.")

Libraries imported successfully.


#### Step 1 - Loading Member 1 Datasets

Three structured datasets prepared by Member 1 are loaded into the project.

These datasets will later be linked with SEC filings using the company ticker,
CIK and reporting year.

In [2]:
# Loading Datasets

company_mapping = pd.read_csv("../data/processed/company_mapping.csv")
processed_data = pd.read_csv("../data/processed/processed_financial_data.csv")
selected_companies = pd.read_csv("../data/processed/selected_companies.csv")

print("All datasets loaded successfully.")

All datasets loaded successfully.


#### Step 2 - Standardizing Column Names

To avoid issues caused by inconsistent capitalization or spaces, all column
names are converted to lowercase and leading/trailing spaces are removed.

In [3]:
# Standardize Column Names

company_mapping.columns = company_mapping.columns.str.strip().str.lower()
processed_data.columns = processed_data.columns.str.strip().str.lower()
selected_companies.columns = selected_companies.columns.str.strip().str.lower()

print("Column names standardized.")

Column names standardized.


#### Step 3 - Verifying Dataset Columns

Inspecting the available columns ensures that the datasets contain the expected information before any preprocessing begins.

In [4]:
# Verify Dataset Columns

print("Company Mapping Columns")
print(company_mapping.columns.tolist())

print("\nProcessed Financial Columns")
print(processed_data.columns.tolist()[:20])

print("\nSelected Companies Columns")
print(selected_companies.columns.tolist())

Company Mapping Columns
['ticker', 'company_name', 'cik', 'sector', 'year']

Processed Financial Columns
['ticker', 'revenue', 'revenue growth', 'cost of revenue', 'gross profit', 'r&d expenses', 'sg&a expense', 'operating expenses', 'operating income', 'interest expense', 'earnings before tax', 'income tax expense', 'net income - non-controlling int', 'net income - discontinued ops', 'net income', 'preferred dividends', 'net income com', 'eps', 'eps diluted', 'weighted average shs out']

Selected Companies Columns
['ticker', 'cik_str', 'title', 'sector']


#### Step 4 - Dataset Dimensions

The dimensions of each dataset are verified to ensure that all expected records
have been loaded successfully.

In [5]:
# Dataset Dimensions

print("="*60)
print("DATASET DIMENSIONS")
print("="*60)
print("Company Mapping     :", company_mapping.shape)
print("Processed Financial :", processed_data.shape)
print("Selected Companies  :", selected_companies.shape)

DATASET DIMENSIONS
Company Mapping     : (110, 5)
Processed Financial : (110, 217)
Selected Companies  : (22, 4)


#### Step 5 - Dataset Preview

The first few rows of each dataset are displayed for manual inspection.

In [6]:
display(company_mapping.head())
display(processed_data.head())
display(selected_companies.head())

,ticker,company_name,cik,sector,year
0,APTV,Aptiv PLC,1521332,Consumer Cyclical,2014
1,APTV,Aptiv PLC,1521332,Consumer Cyclical,2015
2,APTV,Aptiv PLC,1521332,Consumer Cyclical,2016
3,APTV,Aptiv PLC,1521332,Consumer Cyclical,2017
4,APTV,Aptiv PLC,1521332,Consumer Cyclical,2018


,ticker,revenue,revenue growth,cost of revenue,gross profit,r&d expenses,sg&a expense,operating expenses,operating income,interest expense,...,r&d expense growth,sg&a expenses growth,sector,2015 price var [%],class,year,2016 price var [%],2017 price var [%],2018 price var [%],2019 price var [%]
0,POST,2.411100e+09,1.3316,1.789900e+09,6.212000e+08,0.0,459500000.0,8.644000e+08,-243200000.0,183700000.0,...,0.0000,0.5409,Consumer Defensive,44.361250,1,2014,NaN,NaN,NaN,NaN
1,JBSS,7.786220e+08,0.0603,6.557570e+08,1.228650e+08,0.0,77510000.0,7.586900e+07,46996000.0,4354000.0,...,0.0000,-0.0106,Consumer Defensive,35.946195,1,2014,NaN,NaN,NaN,NaN
2,AXTA,4.391500e+09,0.1015,2.897200e+09,1.494300e+09,49500000.0,991500000.0,1.124800e+09,369500000.0,217700000.0,...,0.2222,-0.0472,Basic Materials,1.562499,1,2014,NaN,NaN,NaN,NaN
3,UAN,2.986650e+08,-0.0773,1.981590e+08,1.005060e+08,0.0,17703000.0,1.770300e+07,82803000.0,6783000.0,...,0.0000,-0.1600,Basic Materials,-9.895490,0,2014,NaN,NaN,NaN,NaN
4,PCRX,1.976680e+08,1.3105,7.744000e+07,1.202280e+08,18731000.0,106662000.0,1.253930e+08,-5165000.0,8278000.0,...,-0.1312,0.7064,Healthcare,-15.837349,0,2014,NaN,NaN,NaN,NaN


,ticker,cik_str,title,sector
0,APTV,1521332,Aptiv PLC,Consumer Cyclical
1,ARTNA,863110,ARTESIAN RESOURCES CORP,Utilities
2,ASPS,1462418,ALTISOURCE PORTFOLIO SOLUTIONS S.A.,Industrials
3,AXTA,1616862,Axalta Coating Systems Ltd.,Basic Materials
4,BRN,10048,BARNWELL INDUSTRIES INC,Energy


#### Step 6 - Dataset Information

The structure of each dataset is inspected including:
- Number of rows
- Data types
- Missing values

In [7]:
company_mapping.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110 entries, 0 to 109
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   ticker        110 non-null    object
 1   company_name  110 non-null    object
 2   cik           110 non-null    int64 
 3   sector        110 non-null    object
 4   year          110 non-null    int64 
dtypes: int64(2), object(3)
memory usage: 4.4+ KB


In [8]:
processed_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110 entries, 0 to 109
Columns: 217 entries, ticker to 2019 price var [%]
dtypes: float64(213), int64(2), object(2)
memory usage: 186.6+ KB


In [9]:
selected_companies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22 entries, 0 to 21
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   ticker   22 non-null     object
 1   cik_str  22 non-null     int64 
 2   title    22 non-null     object
 3   sector   22 non-null     object
dtypes: int64(1), object(3)
memory usage: 836.0+ bytes


#### Step 7 - Data Validation

The datasets are validated to ensure that they satisfy the project
requirements.

In [10]:
# Data Validation

print("="*60)
print("DATA VALIDATION")
print("="*60)

print("Unique Companies :", company_mapping["ticker"].nunique())
print("Unique CIKs :", company_mapping["cik"].nunique())

years = sorted(company_mapping["year"].astype(int).unique())

print("Years :", years)
print("Number of Sectors :", company_mapping["sector"].nunique())
print("Missing Values :", company_mapping.isnull().sum().sum())

DATA VALIDATION
Unique Companies : 22
Unique CIKs : 22
Years : [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018)]
Number of Sectors : 11
Missing Values : 0


#### Step 8 - Duplicate Checking

Duplicate records may negatively affect downstream analysis.
Therefore, duplicate rows are checked in each dataset.

In [11]:
# Duplicate Check

print("="*60)
print("DUPLICATE CHECK")
print("="*60)

print("Company Mapping :", company_mapping.duplicated().sum())
print("Processed Data :", processed_data.duplicated().sum())
print("Selected Companies :", selected_companies.duplicated().sum())

DUPLICATE CHECK
Company Mapping : 0
Processed Data : 0
Selected Companies : 0


In [12]:
assert company_mapping.duplicated().sum() == 0
assert selected_companies.duplicated().sum() == 0

#### Step 9 - Missing Values

Missing values are inspected on a column-by-column basis.

In [13]:
company_mapping.isnull().sum()

ticker          0
company_name    0
cik             0
sector          0
year            0
dtype: int64

In [14]:
processed_data.isnull().sum().sort_values(ascending=False).head(20)

2019 price var [%]              88
2018 price var [%]              88
2017 price var [%]              88
2016 price var [%]              88
2015 price var [%]              88
enterprise value over ebitda     0
free cash flow yield             0
earnings yield                   0
ev to free cash flow             0
ev to operating cash flow        0
pb ratio                         0
ev to sales                      0
ptb ratio                        0
debt to assets                   0
pfcf ratio                       0
pocf ratio                       0
debt to equity                   0
current ratio                    0
net debt to ebitda               0
pe ratio                         0
dtype: int64

In [15]:
selected_companies.isnull().sum()

ticker     0
cik_str    0
title      0
sector     0
dtype: int64

#### Step 10 - CIK Mapping Validation

Each company ticker should be associated with exactly one Central Index Key (CIK).

The CIK is the unique identifier used by the U.S. Securities and Exchange Commission (SEC) to identify companies. This mapping is critical because the SEC filing collection process in Notebook 2 will use the CIK to retrieve the correct 10-K filings.

This validation ensures that every selected company has a unique and consistent CIK throughout the dataset.

In [16]:
# CIK Mapping Validation

print("=" * 60)
print("CIK MAPPING VALIDATION")
print("=" * 60)

mapping_check = company_mapping.groupby("ticker")["cik"].nunique()

display(mapping_check)

if (mapping_check == 1).all():
    print("\nAll companies have a unique CIK.")
else:
    print("\nWarning: Some companies are mapped to multiple CIKs.")

CIK MAPPING VALIDATION


ticker
APTV     1
ARTNA    1
ASPS     1
AXTA     1
BRN      1
CLDX     1
FCNCA    1
GEOS     1
GTY      1
INTT     1
JBSS     1
MAYS     1
NOVT     1
OPHC     1
PCRX     1
POST     1
SR       1
T        1
TMUS     1
UAN      1
ULBI     1
VAC      1
Name: cik, dtype: int64


All companies have a unique CIK.


#### Step 11 - Company-Year Validation

Each selected company should have one observation for each year
from 2014 to 2018.

This validation confirms that the structured dataset is complete.

In [17]:
company_years = company_mapping.groupby("ticker")["year"].count()
display(company_years)

ticker
APTV     5
ARTNA    5
ASPS     5
AXTA     5
BRN      5
CLDX     5
FCNCA    5
GEOS     5
GTY      5
INTT     5
JBSS     5
MAYS     5
NOVT     5
OPHC     5
PCRX     5
POST     5
SR       5
T        5
TMUS     5
UAN      5
ULBI     5
VAC      5
Name: year, dtype: int64

In [18]:
expected_years = 5

invalid = company_years[company_years != expected_years]

if len(invalid) == 0:
    print("All companies contain 5 reporting years.")
else:
    print("Some companies have missing years:")
    display(invalid)

All companies contain 5 reporting years.


#### Step 12 - Project Summary

In [19]:
print("="*70)
print("PROJECT SUMMARY")
print("="*70)

print(f"Selected Companies        : {selected_companies.shape[0]}")
print(f"Company-Year Records      : {company_mapping.shape[0]}")
print(f"Unique Companies          : {company_mapping['ticker'].nunique()}")
print(f"Unique CIKs               : {company_mapping['cik'].nunique()}")
print(f"Financial Features        : {processed_data.shape[1]}")
print(f"Available Sectors         : {company_mapping['sector'].nunique()}")
print(f"Reporting Years           : {years}")
print()

print("Notebook Status")
print("----------------")
print("✓ Datasets Loaded")
print("✓ Column Names Standardized")
print("✓ Dataset Structure Verified")
print("✓ No Critical Missing Values")
print("✓ Duplicate Check Completed")
print("✓ Company-Year Mapping Verified")
print()

print("STATUS : READY FOR NOTEBOOK 2")

PROJECT SUMMARY
Selected Companies        : 22
Company-Year Records      : 110
Unique Companies          : 22
Unique CIKs               : 22
Financial Features        : 217
Available Sectors         : 11
Reporting Years           : [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018)]

Notebook Status
----------------
✓ Datasets Loaded
✓ Column Names Standardized
✓ Dataset Structure Verified
✓ No Critical Missing Values
✓ Duplicate Check Completed
✓ Company-Year Mapping Verified

STATUS : READY FOR NOTEBOOK 2


#### Conclusion

The structured datasets prepared by Member 1 have been successfully
loaded and validated.

The project contains:

- 22 selected companies
- 110 company-year observations
- Financial indicators
- Company-to-CIK mapping

These datasets are now ready to be linked with SEC 10-K filings in
Notebook 2.

The next notebook will focus on collecting SEC filings for the selected
companies and years, which will become the narrative knowledge base for
the Retrieval-Augmented Generation (RAG) pipeline.